In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Getting started with Interactions API

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_interactions_api.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fgetting-started%2Fintro_interactions_api.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/getting-started/intro_interactions_api.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_interactions_api.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_interactions_api.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_interactions_api.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_interactions_api.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_interactions_api.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_interactions_api.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Authors |
| --- |
| [Philipp Schmid](https://github.com/philschmid) |
| [Guillaume Vernade](https://github.com/Giom-V) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions?ua=chat) is a unified interface for building with Gemini models and agents. It simplifies the development of agentic applications by handling server-side state management, tool orchestration, and long-running tasks.

With a single endpoint, you can:

- Interact with Gemini models for text, image, and audio generation
- Build multi-turn conversations without managing history client-side
- Call custom functions and built-in tools like Google Search
- Run specialized agents like Deep Research for complex tasks

<a name="setup"></a>
## Setup

### Install Google Gen AI SDK and other required packages


In [ ]:
%pip install --upgrade --quiet google-genai

### Import Libraries

Import the required system and visual display components. We will also import the Pydantic library for validating structured outputs.

In [ ]:
import base64
import os
import sys
import warnings

import requests
from IPython.display import Image, Markdown, display
from google.genai.interactions import (
    AudioContent,
    DocumentContent,
    Function,
    GenerationConfig,
    GoogleSearch,
    ImageContent,
    TextContent,
    TextResponseFormat,
    VideoContent,
)
from pydantic import BaseModel

warnings.filterwarnings("ignore")

### Authenticate your notebook environment

If you are running this notebook in **Google Colab**, run the cell below to authenticate your account.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
from google import genai
from google.genai import types

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "global" # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(timeout=20000),
)

### Choose a model

Select the model you want to use in this guide. You can either select one from the list or enter a model name manually. 

For a full overview of all Gemini models, check the [documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/google-models).

The Interactions API in Agent Platform is supported by all Gemini 3 models.

In [ ]:
# fmt: off
MODEL_ID = "gemini-3.8-flash"  # @param ["gemini-3.5-flash-lite", "gemini-3.1-pro-preview", "gemini-3.8-flash"] {"allow-input":true, isTemplate: true}
# fmt: on

## Create an interaction

At its simplest, the Interactions API works like a standard chat completion. You provide a model and an input string. By default, interactions are stored (`store=True`), allowing you to reference them later.

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    system_instruction="You are an expert in physics and science communication.",
    input="Explain quantum entanglement in one sentence.",
)

display(Markdown(interaction.output_text))

### Streaming output

For user-facing or chat-based interfaces, use streaming to receive content chunks immediately as they are being computed.

In [ ]:
prompt = "Count from 1 to 100."

stream = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    stream=True,
)
for event in stream:
    if event.event_type == "step.delta" and event.delta.type == "text":
        print(event.delta.text, end="", flush=True)

## Stateful Multi-turn Conversations

One of the most powerful features of this API is **server-side state management**. You do not need to append messages to a list and send the full history back to the server every time.

Use `previous_interaction_id` to continue a conversation.

Note: Only conversation history is preserved. Parameters like `tools` or `generation_config` are interaction-scoped and must be re-declared if needed.

In [ ]:
turn_1 = client.interactions.create(
    model=MODEL_ID, input="My name is Alice and I am a software engineer."
)
display(Markdown(f"Turn 1 ID: `{turn_1.id}`"))

turn_2 = client.interactions.create(
    model=MODEL_ID,
    input="What is my job?",
    previous_interaction_id=turn_1.id,
)
display(Markdown(f"Turn 2 ID: `{turn_2.id}`"))
display(Markdown(turn_2.output_text))

### Forking Conversations

Because state is managed by ID, you can "fork" a conversation by referencing an older interaction ID with a different prompt.

In [ ]:
# Branch off from Turn 1 with a different topic
turn_2_fork = client.interactions.create(
    model=MODEL_ID,
    input="What is my name?",
    previous_interaction_id=turn_1.id,
)
display(Markdown(f"Turn 1 ID: `{turn_1.id}`"))
display(Markdown(f"Turn 2 Fork ID: `{turn_2_fork.id}`"))
display(Markdown(turn_2_fork.output_text))

## Configure model parameters and thinking

You can include parameter values in each call you send to a model using `generation_config`.

Gemini models support configurable **thinking levels** that control how much internal reasoning the model performs before responding. Higher thinking levels produce better results for complex tasks but use more tokens and take longer.

To see the model thoughts, you need to check the `steps` that are of type `thought`. This will not give you the full details of what the model "thought", but you'll have a summary to help you understand how the model arrived to its conclusion.

| Thinking Level | Description | Recommended Use Case |
| :--- | :--- | :--- |
| `LOW` | Lightweight logical checks with high execution speed and low token cost. | Moderate concepts comparison, editing. |
| `MEDIUM` (Default) | Highly balanced reasoning effort. | Standard problem solving, standard logical deduction. |
| `HIGH` (Dynamic) | Maximizes internal chain-of-thought loops before first-token output. | Advanced math, logic puzzles, complex code generation. |

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Tell me how the internet works, but pretend I'm a puppy who understands only dog-related analogies.",
    generation_config=GenerationConfig(max_output_tokens=500, thinking_level="low"),
)

display(Markdown(interaction.output_text))

## Multimodal Prompts

Gemini models have strong multimodal understanding capabilities. You can include text PDF documents, images, audio, and videos in your prompt requests and get text or code responses.

### Images

This example uses an image via an URI.

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        ImageContent(
            uri="https://storage.googleapis.com/generativeai-downloads/images/generated_elephants_giraffes_zebras_sunset.jpg"
        ),
        TextContent(text="What do you see in this image? Describe it briefly."),
    ],
)

display(Markdown(interaction.output_text))

You can also download an image and send it as base64-encoded data. This is useful when you need to preprocess the image or when the URL is not publicly accessible:

In [ ]:
IMG = "https://storage.googleapis.com/generativeai-downloads/data/jetpack.png"

img_bytes = requests.get(IMG, timeout=10).content
display(Image(img_bytes, width=400))

In [ ]:
img_b64 = base64.b64encode(img_bytes).decode()

response = client.interactions.create(
    model=MODEL_ID,
    input=[
        ImageContent(data=img_b64, mime_type="image/png"),
        TextContent(text="Write a short and engaging blog post based on this picture."),
    ],
    generation_config=GenerationConfig(thinking_level="low"),
)

display(Markdown(response.output_text))

### PDF Document Analysis

In this example, we will summarize a document hosted on Google Cloud Storage.

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        DocumentContent(
            uri="gs://cloud-samples-data/generative-ai/pdf/1706.03762v7.pdf",
            mime_type="application/pdf",
        ),
        TextContent(text="Summarize the document."),
    ],
    generation_config=GenerationConfig(thinking_level="low"),
)

display(Markdown(interaction.output_text))

### Audio Analysis 

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        AudioContent(
            uri="https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/pixel.mp3",
            mime_type="audio/mpeg",
        ),
        TextContent(text="Write a summary of this podcast episode."),
    ],
    generation_config=GenerationConfig(thinking_level="low"),
)

display(Markdown(data=interaction.output_text))

### Video Analysis (YouTube)

This example uses [Agentic video processing](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/capabilities/video-understanding#agentic-video-processing).

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        VideoContent(
            uri="https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/pixel.mp3",
            mime_type="video/webm",
            processing="agentic",
        ),
        TextContent(text="At what point in the video is Harry Potter shown?"),
    ],
    generation_config=GenerationConfig(thinking_level="low"),
)
display(Markdown(data=interaction.output_text))

## Image Generation

In [ ]:
interaction = client.interactions.create(
    model="gemini-3.1-flash-image",
    input="Generate an image of a cat wearing a top hat in a library.",
)

display(Image(data=base64.b64decode(interaction.output_image.data)))

## Structured output

Enforce a schema constraint to guarantee that the model outputs structured JSON matching a predefined format.

In [ ]:
class CountryInfo(BaseModel):
    name: str
    population: int
    capital: str
    continent: str
    gdp: int
    official_language: str
    total_area_sq_mi: int


prompt = "Give me information for the United States."

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    response_format=TextResponseFormat(
        mime_type="application/json",
        schema=CountryInfo.model_json_schema(),
    ),
)

country = CountryInfo.model_validate_json(interaction.output_text)
print(country)

## Tool use

Tools extend the model's capabilities by letting it call external functions or
services. The API includes ready-to-use tools, like Google Search or lets you define custom tools as a JSON schema. The model decides when to call them based on the conversation:

### Built-in tools

#### Google Search

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Who won the 2024 Nobel Prize in Physics?",
    tools=[GoogleSearch()],
)

display(Markdown(interaction.output_text))

Other built-in tools include [Code Execution](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/sandbox/code-execution-overview) and [Computer Use](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/sandbox/computer-use).

### Function calling


In [ ]:
# Define a function
def get_weather(city: str) -> dict[str, str]:
    """Gets the weather in a city."""
    if "london" in city.lower():
        return {"temperature": "23°C", "condition": "Rainy"}
    if "new york" in city.lower():
        return {"temperature": "30°C", "condition": "Sunny"}
        return "Sunny"
    return {"temperature": "25°C", "condition": "Cloudy"}


# Define a tool
weather_tool = Function(
    name="get_weather",
    description="Get current weather for a location",
    parameters={
        "type": "object",
        "properties": {"location": {"type": "string", "description": "City name"}},
        "required": ["location"],
    },
)


# Send request with tool
interaction = client.interactions.create(
    model=MODEL_ID, input="What's the weather in Tokyo?", tools=[weather_tool]
)

# Handle tool call
for step in interaction.steps:
    if step.type == "function_call":
        # Execute your function
        result = get_weather(step.arguments["location"])

        # Return result to model
        interaction = client.interactions.create(
            model=MODEL_ID,
            previous_interaction_id=interaction.id,
            input=[
                {
                    "type": "function_result",
                    "name": step.name,
                    "call_id": step.id,
                    "result": result,
                }
            ],
        )
        print(interaction.output_text)

## Agents & Long-Running Tasks

Beyond models, the Interactions API provides access to specialized agents.
Deep Research executes multi-step research tasks, synthesizing information from
multiple sources into comprehensive reports.

Agents run asynchronously with `background=True`. Poll the interaction status
to retrieve results:

For more details, see [Deep Research](https://docs.cloud.google.com/gemini-enterprise-agent-platform/agents/google/deep-research).

In [ ]:
import time

agent_interaction = client.interactions.create(
    agent="deep-research-preview-04-2026",  # Note: use 'agent', not 'model'
    input="Research the history of the Google TPUs with a focus on 2025 specs.",
    background=True,
)


# Poll for completion
while True:
    status_check = client.interactions.get(agent_interaction.id)
    print(f"Status: {status_check.status}")

    if status_check.status == "completed":
        print("\n--- Final Report ---\n")
        print(status_check.output_text)
        break
    elif status_check.status in ["failed", "cancelled"]:
        print("Agent failed.")
        break

    time.sleep(10)